In [17]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import sympy as sp 
import cvxpy as cp
import meanvarainceblackletterman as mvo2

In [18]:
mvo2.x_bar = 0.10          # price goes from flat fee to linear fee
mvo2.x_low_sell = 0.001
mvo2.x_high_sell = 0.99
mvo2.x_low_buy = 0.001
mvo2.x_high_buy = 0.99
c0 = 0.001   # flat fee estimate of 1% of position
c1 = 0.005  # 1% of position as linear cost

### Her har jeg hentet to års data fra Yahoo-finance for eksemplets skyld

In [19]:
prices = yf.download(tickers, period="2y", auto_adjust=True, progress=False)["Close"]
prices = prices.dropna(axis=1, thresh=int(0.95 * len(prices)))  # drop tickers med for meget manglende historik
prices = prices.dropna()

daily_rets = prices.pct_change().dropna()

# Hvis nogle tickers blev droppet pga. manglende data, opdater listen så den matcher
tickers = list(daily_rets.columns)
n = len(tickers)

print(f"Rene data for {n} aktiver, {len(daily_rets)} handelsdage.")
daily_rets.head()

Rene data for 10 aktiver, 501 handelsdage.


Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,TSLA,XOM
Date,,,,,,,,,,
2024-08-29,0.014570,0.007728,-0.006570,0.001891,0.004157,0.002787,0.006137,-0.063849,0.002576,0.013817
2024-08-30,-0.003438,0.037067,0.009890,0.009925,0.011656,0.005963,0.009731,0.015137,0.037958,-0.001608
2024-09-03,-0.027205,-0.012605,-0.036847,0.007838,-0.020018,-0.018319,-0.018459,-0.095250,-0.016393,-0.020943
2024-09-04,-0.008619,-0.016567,-0.005783,0.001196,-0.004403,0.001915,-0.001319,-0.016574,0.041833,-0.012211
2024-09-05,0.006928,0.026308,0.005050,-0.014161,-0.007751,0.008035,-0.001247,0.009415,0.049041,-0.007803


# For robusthed bruger jeg Ledoit-Wolf

In [20]:
from sklearn.covariance import LedoitWolf
def shrink_cov(daily_rets_slice, annualize=252):
    lw = LedoitWolf().fit(daily_rets_slice.values)
    return lw.covariance_ * annualize

Sigma_matrix = shrink_cov(daily_rets)
Sigma = cp.psd_wrap(Sigma_matrix)

print(f"Kovariansmatrix beregnet: {Sigma_matrix.shape[0]}×{Sigma_matrix.shape[1]}")

Kovariansmatrix beregnet: 10×10


In [25]:
import yfinance as yf
## EKSEMPEL: Sådan indtaster du dine egne finansielle holdninger (views)
## i modellen — udfyld kun de markerede steder, resten kører automatisk.

# --------------------------------------------------------------------
# TRIN 1: Dine 10 aktiver, i den rækkefølge du vil arbejde med dem.
# Husk nummereringen — aktiv nr. 0 er "AAPL", nr. 1 er "MSFT", osv.
# Det er VIGTIGT, fordi du refererer til aktiver via deres nummer nedenfor.
# --------------------------------------------------------------------
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "JPM", "XOM", "JNJ"]
n = len(tickers)   # = 10, regnes automatisk ud fra listen ovenfor

for i, t in enumerate(tickers):
    print(f"  Aktiv {i}: {t}")


# --------------------------------------------------------------------
# TRIN 2: Skriv dine views her.
# Et "view" er en konkret holdning, du har til fremtidigt afkast.
# Der findes to slags views:
#
#   A) ABSOLUT view: "Jeg tror Aktiv X vil give Y% i afkast"
#   B) RELATIVT view: "Jeg tror Aktiv X vil klare sig Y%-point bedre end Aktiv Z"
#
# Under hvert view skal du udfylde TRE ting:
#   - hvilke(t) aktiv/aktiver det handler om (via deres nummer fra Trin 1)
#   - dit forventede tal (Q)
#   - hvor SIKKER du er på det (Omega) — lavt tal = meget sikker, højt tal = usikker
# --------------------------------------------------------------------
lamb=0.71 #Jeg vælger en tilfældig lambda værdi


# --- View 1 (ABSOLUT): "Jeg tror NVIDIA (aktiv 4) vil give 20% i afkast" ---
view1_asset = 4          # NVDA er nummer 4 på listen ovenfor
view1_forventning = 0.20  # 20% forventet afkast
view1_sikkerhed = 0.0002    # hvor usikker du er (se forklaring nedenfor)

# --- View 2 (ABSOLUT): "Jeg tror Exxon (aktiv 8) vil give 5% i afkast" ---
view2_asset = 8
view2_forventning = 0.05
view2_sikkerhed = 0.00015   # mere sikker på denne end på NVIDIA-viewet

# --- View 3 (RELATIVT): "Jeg tror Apple (aktiv 0) vil slå Johnson & Johnson (aktiv 9) med 8%-point" ---
view3_asset_a = 0    # AAPL — den vi tror klarer sig BEDST
view3_asset_b = 9    # JNJ  — den vi tror klarer sig DÅRLIGST i sammenligningen
view3_forskel = 0.08  # AAPL forventes at slå JNJ med 8 procentpoint
view3_sikkerhed = 0.0003


# --------------------------------------------------------------------
# TRIN 3: Herfra oversætter koden automatisk dine views til det format,
# modellen forstår (P, Q_views, Omega). DU BEHØVER IKKE ÆNDRE NOGET HERUNDER
# — det sker automatisk ud fra det, du skrev i Trin 2.
# --------------------------------------------------------------------
K = 3  # antal views, du har skrevet ovenfor (ret dette tal hvis du tilføjer/fjerner views)

P = np.zeros((K, n))          # "hvilke aktiver hvert view handler om"
Q_views = np.zeros(K)          # "dit forventede tal per view"
Omega_diag = np.zeros(K)       # "din usikkerhed per view"

# View 1 → række 0 i P
P[0, view1_asset] = 1
Q_views[0] = view1_forventning
Omega_diag[0] = view1_sikkerhed

# View 2 → række 1 i P
P[1, view2_asset] = 1
Q_views[1] = view2_forventning
Omega_diag[1] = view2_sikkerhed

# View 3 → række 2 i P (relativt view: +1 på den bedste, -1 på den dårligste)
P[2, view3_asset_a] = 1
P[2, view3_asset_b] = -1
Q_views[2] = view3_forskel
Omega_diag[2] = view3_sikkerhed

Omega = np.diag(Omega_diag)   # Omega skal være en "diagonal matrix" — dette laver den automatisk


# --------------------------------------------------------------------
# TRIN 4: Kør modellen. Igen — intet at ændre her.
# --------------------------------------------------------------------
rets_slice = daily_rets.iloc[:, :n]
Sigma_matrix = shrink_cov(rets_slice)       # samme robuste kovarians-beregning som hele tiden
Sigma = cp.psd_wrap(Sigma_matrix)

vols = np.sqrt(np.diag(Sigma_matrix))
w_mkt = (1 / vols) / np.sum(1 / vols)       # ERSTAT med rigtige markedsværdi-vægte, når I har dem

tau = 0.05

mu_bl = mvo2.black_litterman_returns(Sigma_matrix, w_mkt, lamb, P, Q_views, Omega, tau)

print("\nForventet afkast pr. aktiv, EFTER dine views er indregnet:")
for i, t in enumerate(tickers):
    print(f"  {t}: {mu_bl[i]*100:5.2f}%")

weights, z_bin = mvo2.meanvariance_frontier(lamb, Sigma, mu_bl, c0, c1, n)

print("\nForeslåede porteføljevægte:")
for i, t in enumerate(tickers):
    if weights[i] > 0.001:
        print(f"  {t}: {weights[i]*100:5.1f}%")

  Aktiv 0: AAPL
  Aktiv 1: MSFT
  Aktiv 2: GOOGL
  Aktiv 3: AMZN
  Aktiv 4: NVDA
  Aktiv 5: META
  Aktiv 6: TSLA
  Aktiv 7: JPM
  Aktiv 8: XOM
  Aktiv 9: JNJ

Forventet afkast pr. aktiv, EFTER dine views er indregnet:
  AAPL:  9.05%
  MSFT:  8.90%
  GOOGL:  6.48%
  AMZN:  1.40%
  NVDA: 18.86%
  META:  9.07%
  TSLA:  5.47%
  JPM:  9.63%
  XOM:  5.14%
  JNJ:  1.29%

Foreslåede porteføljevægte:
  AAPL:   9.5%
  GOOGL:   2.3%
  AMZN:  19.4%
  NVDA:  60.5%
  TSLA:   8.2%
